# Lesson 4: Introduction to SQL

**Week 1 · Data Engineering Course**

---

**Topics:** `SELECT` · `WHERE` · `ORDER BY` · `LIMIT` · `DISTINCT`

**Prerequisites:** Lab 1 complete — the `employees` table must already be loaded into `de_course`.

---

## How to use this notebook

This lesson is a **reference guide** — all SQL runs in **pgAdmin**, not here.

For each example:
1. Open **pgAdmin** → connect to `de_course` → **Tools → Query Tool**
2. Copy the SQL from the grey code block below
3. Paste it into the pgAdmin editor
4. Press **F5** to run and see results in the **Data Output** tab

> **Tip:** Keep this notebook open on one half of your screen and pgAdmin on the other.

## What is SQL?

**SQL** (Structured Query Language) is the language used to communicate with relational databases.
You describe *what* data you want, and the database figures out *how* to retrieve it.

- **Declarative** — say *what*, not *how*
- **Universal** — works on PostgreSQL, MySQL, BigQuery, Snowflake, and nearly every other database
- **Essential** — you will write SQL every single day as a data engineer

## The Database Model

Data is stored in **tables** — grids of rows and columns, like a spreadsheet.

- Each **column** has a name and a data type (text, number, date, boolean…)
- Each **row** is one record — one employee, one order, one event
- A **primary key** is a column that uniquely identifies each row — no two rows can share the same value, and it cannot be empty. In our `employees` table, `employee_id` is the primary key.

> **What is NULL?** `NULL` means a value is *missing* or *unknown*. It is not zero and not an empty string. A salary of NULL means "no salary on record", which is different from a salary of 0.

Run this in pgAdmin to preview the `employees` table:

```sql
SELECT * FROM employees LIMIT 5;
```

And to confirm the row count:

```sql
SELECT COUNT(*) AS total_rows FROM employees;
```

---

## 4.1 SELECT — Retrieve Data

```sql
SELECT column1, column2, ...
FROM table_name;
```

`*` selects **all columns** — fine for exploration, avoid in production (breaks when the schema changes):

```sql
SELECT *
FROM employees
LIMIT 3;
```

---

Select **specific columns** (preferred — explicit and readable):

```sql
SELECT first_name, last_name, salary
FROM employees;
```

---

**Column aliases** — rename output columns with `AS`:

```sql
SELECT
    first_name AS name,
    salary     AS annual_salary
FROM employees
LIMIT 5;
```

---

**Computed columns** — do arithmetic directly in `SELECT`:

```sql
SELECT
    first_name,
    salary,
    ROUND(salary / 12.0, 2) AS monthly_salary
FROM employees
ORDER BY salary DESC
LIMIT 5;
```

---

## 4.2 WHERE — Filter Rows

```sql
SELECT ...
FROM table_name
WHERE condition;
```

Only rows where the condition is `true` are returned.

### Comparison operators

| Operator | Meaning | Example |
|----------|---------|----------|
| `=` | Equals | `department = 'Engineering'` |
| `<>` or `!=` | Not equals | `department <> 'Sales'` |
| `>` | Greater than | `salary > 80000` |
| `<` | Less than | `salary < 80000` |
| `>=` | Greater or equal | `salary >= 95000` |
| `<=` | Less or equal | `salary <= 72000` |

> Use **single quotes** for text values: `'Engineering'`, not `"Engineering"`.

---

**Filter by text:**

```sql
SELECT first_name, last_name, department
FROM employees
WHERE department = 'Engineering';
```

**Filter by number:**

```sql
SELECT first_name, salary
FROM employees
WHERE salary > 90000
ORDER BY salary DESC;
```

### AND / OR / NOT

**AND** — both conditions must be true:

```sql
SELECT first_name, department, salary
FROM employees
WHERE department = 'Engineering'
  AND salary > 100000;
```

**OR** — either condition can be true:

```sql
SELECT first_name, department
FROM employees
WHERE department = 'Engineering'
   OR department = 'Marketing'
ORDER BY department, first_name;
```

**NOT** — negate a condition:

```sql
SELECT first_name, department
FROM employees
WHERE NOT department = 'Sales'
ORDER BY department;
```

---

> **AND runs before OR — always use parentheses when mixing them.**
>
> `WHERE a AND b OR c` is evaluated as `WHERE (a AND b) OR c`, which may not be what you mean.
> Make your intent explicit:
>
> ```sql
> -- Engineering employees who either earn a lot OR were hired early
> WHERE department = 'Engineering'
>   AND (salary > 100000 OR hire_date < '2020-01-01')
> ```
>
> Without the parentheses, the query would return all Engineering employees with salary > 100k,
> PLUS any employee (from any department) hired before 2020.

### BETWEEN — range check

`BETWEEN` is **inclusive** on both ends (same as `>= AND <=`):

```sql
SELECT first_name, salary
FROM employees
WHERE salary BETWEEN 70000 AND 80000
ORDER BY salary;
```

### IN — match a list of values

Cleaner than chaining multiple `OR` conditions:

```sql
SELECT first_name, department
FROM employees
WHERE department IN ('Engineering', 'Marketing')
ORDER BY department, first_name;
```

### LIKE — pattern matching

- `%` matches any sequence of characters (including none)
- `_` matches exactly one character

```sql
-- Names starting with A, B, or C
SELECT first_name, last_name
FROM employees
WHERE first_name LIKE 'A%'
   OR first_name LIKE 'B%'
   OR first_name LIKE 'C%'
ORDER BY first_name;
```

```sql
-- Names containing 'li' anywhere
SELECT first_name
FROM employees
WHERE first_name LIKE '%li%';
```

### IS NULL / IS NOT NULL

> **Never write `WHERE column = NULL`** — that always returns zero rows because NULL is not equal to anything, not even another NULL.
> Always use `IS NULL` or `IS NOT NULL`.

```sql
SELECT first_name, hire_date
FROM employees
WHERE hire_date IS NOT NULL
LIMIT 5;
```

---

## 4.3 ORDER BY — Sort Results

```sql
SELECT ...
FROM ...
ORDER BY column1 [ASC | DESC], column2 [ASC | DESC];
```

Default is ascending (`ASC`). Use `DESC` to reverse.

---

**Sort by salary, highest first:**

```sql
SELECT first_name, salary
FROM employees
ORDER BY salary DESC;
```

---

**Sort by multiple columns** — department A→Z, then salary high→low within each department:

```sql
SELECT department, first_name, salary
FROM employees
ORDER BY department ASC, salary DESC;
```

---

**Sort by date** — oldest hire first:

```sql
SELECT first_name, hire_date, department
FROM employees
ORDER BY hire_date ASC;
```

---

## 4.4 LIMIT — Restrict Row Count

```sql
SELECT ...
FROM ...
LIMIT n;
```

> **Always use `LIMIT` when exploring an unfamiliar table.** Production tables can have billions of rows — a query without a limit can run for minutes and return more data than your machine can handle.

---

**Peek at the first 5 rows:**

```sql
SELECT * FROM employees LIMIT 5;
```

**Top 3 highest earners:**

```sql
SELECT first_name, last_name, salary
FROM employees
ORDER BY salary DESC
LIMIT 3;
```

**3 most recently hired:**

```sql
SELECT first_name, hire_date, department
FROM employees
ORDER BY hire_date DESC
LIMIT 3;
```

---

## 4.5 DISTINCT — Remove Duplicates

`DISTINCT` goes right after `SELECT` and eliminates duplicate rows.

```sql
SELECT DISTINCT column1, column2
FROM table_name;
```

---

**What departments exist in the company?**

```sql
SELECT DISTINCT department
FROM employees
ORDER BY department;
```

Without `DISTINCT` you'd get one row per employee — many repeated department names.

---

**What cities do employees work in?**

```sql
SELECT DISTINCT city
FROM employees
ORDER BY city;
```

**Unique (department, city) combinations:**

```sql
SELECT DISTINCT department, city
FROM employees
ORDER BY department, city;
```

**COUNT DISTINCT — how many unique departments are there?**

```sql
SELECT COUNT(DISTINCT department) AS num_departments
FROM employees;
```

---

## 4.6 Putting It All Together

The clauses always appear in this **fixed order**:

```sql
SELECT   [DISTINCT] column1, column2, ...
FROM     table_name
WHERE    condition
ORDER BY column1 [ASC | DESC]
LIMIT    n;
```

### Full example

> *"Show the top 3 highest-paid active employees outside of Sales, sorted by salary descending."*

```sql
SELECT
    first_name,
    last_name,
    department,
    salary,
    hire_date
FROM employees
WHERE department <> 'Sales'
  AND is_active = TRUE
ORDER BY salary DESC
LIMIT 3;
```

Run this in pgAdmin. You should see Carol Williams, Alice Johnson, and Quinn Martinez at the top.

---

## 4.7 Inspecting the Table Schema in pgAdmin

Before writing queries you often need to know what columns a table has and what types they are.

**Option A — Object Explorer (no SQL needed):**
1. In the left panel expand `de_course → Schemas → public → Tables`
2. Right-click `employees` → **Properties**
3. Click the **Columns** tab

**Option B — SQL query:**

```sql
SELECT column_name, data_type, character_maximum_length
FROM information_schema.columns
WHERE table_name = 'employees'
ORDER BY ordinal_position;
```

### Common PostgreSQL data types

| Type | Description | Example |
|------|-------------|----------|
| `INTEGER` / `INT` | Whole numbers | `42`, `-7` |
| `BIGINT` | Very large whole numbers | `9876543210` |
| `NUMERIC(p,s)` | Exact decimals | `123.45` |
| `FLOAT` / `REAL` | Approximate decimals | `3.14` |
| `VARCHAR(n)` | Text, max n characters | `'Alice'` |
| `TEXT` | Unlimited text | `'Any length'` |
| `BOOLEAN` | True or false | `TRUE`, `FALSE` |
| `DATE` | Calendar date | `2024-01-15` |
| `TIMESTAMP` | Date + time | `2024-01-15 09:30:00` |

---

## Key Takeaways

1. `SELECT` chooses columns; `FROM` chooses the table.
2. `WHERE` filters rows — use comparisons, `IN`, `LIKE`, `BETWEEN`, `IS NULL`.
3. `ORDER BY` sorts results; `ASC` is default, `DESC` reverses.
4. `LIMIT` caps the row count — always use it when exploring.
5. `DISTINCT` removes duplicate rows.
6. Clause order is **fixed**: `SELECT → FROM → WHERE → ORDER BY → LIMIT`.
7. `NULL` means missing — use `IS NULL`, never `= NULL`.
8. AND runs before OR — use parentheses when mixing them.

---

## ✏️ Check Your Understanding

Write a SQL query for each question in pgAdmin's Query Tool, verify it returns the expected result, then paste the working SQL into the cell below each question.

> The cells below are **text cells** — paste SQL as plain text. They will not run.

---

### Q1: Which employees earn more than $75,000?

*(Expected: multiple rows from Engineering, Marketing, Sales)*

### Q2: What are all the unique departments?

*(Expected: 4 rows — Engineering, HR, Marketing, Sales)*

### Q3: Who are the bottom 2 earners in the Marketing department?

*(Expected: 2 rows, lowest salaries in Marketing)*

### Q4: Show all employees hired after January 1, 2021, sorted by hire_date ascending.

*(Expected: employees from 2021 onwards, oldest first)*

### Q5: How many distinct departments are there?

*(Expected: 1 row with value 4)*

---

## Solutions

Try every question before looking here.

---

**Q1 — Employees earning > $75,000**

```sql
SELECT first_name, last_name, department, salary
FROM employees
WHERE salary > 75000
ORDER BY salary DESC;
```

---

**Q2 — Unique departments**

```sql
SELECT DISTINCT department
FROM employees
ORDER BY department;
```

---

**Q3 — Bottom 2 earners in Marketing**

```sql
SELECT first_name, last_name, salary
FROM employees
WHERE department = 'Marketing'
ORDER BY salary ASC
LIMIT 2;
```

---

**Q4 — Hired after 2021-01-01**

```sql
SELECT first_name, last_name, hire_date, department
FROM employees
WHERE hire_date > '2021-01-01'
ORDER BY hire_date ASC;
```

---

**Q5 — Count of distinct departments**

```sql
SELECT COUNT(DISTINCT department) AS num_departments
FROM employees;
```